# 🎯 YOLOv8 Sniper Training: Orange Collector (Optimized for Colab)

Этот ноутбук оптимизирован для обучения модели YOLOv8 в Google Colab. 
Он включает подключение Google Drive для сохранения весов, чтобы не потерять прогресс при отключении сессии, и адаптированные параметры обучения.

### 1. Подключение Google Drive
Подключаем диск, чтобы сохранять туда обученную модель. Иначе при закрытии вкладки всё удалится.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Создаем папку для сохранения результатов на диске
os.makedirs('/content/drive/MyDrive/YOLO_Orange_Sniper', exist_ok=True)

### 2. Подготовка окружения и загрузка репозитория
Устанавливаем нужные библиотеки и скачиваем код.

In [ ]:
!pip install ultralytics opencv-python

!rm -rf /content/drake
!git clone https://github.com/drakeludo/drake.git /content/drake
%cd /content/drake

### 3. Проверка GPU
Убедимся, что нам выдали видеокарту (Среда выполнения -> Сменить среду выполнения -> T4 GPU).

In [ ]:
!nvidia-smi
import torch
print(f"CUDA доступна: {torch.cuda.is_available()}")

### 4. Загрузка датасета
**ВНИМАНИЕ:** В репозитории нет самих картинок. Вам нужно загрузить архив с датасетом (например, `dataset.zip`) на ваш Google Drive в папку `YOLO_Orange_Sniper`.

Раскомментируйте и выполните ячейку ниже, если у вас есть архив с датасетом на диске.

In [ ]:
# !unzip -q /content/drive/MyDrive/YOLO_Orange_Sniper/dataset.zip -d /content/drake/yolo_dataset/

### 5. Создание оптимизированного скрипта обучения
Оригинальный скрипт настроен на 20,000 эпох и 8 воркеров, что не подходит для Colab. Мы создадим новый скрипт с оптимальными параметрами.

In [ ]:
train_script = """
from ultralytics import YOLO
import os
import shutil

# Путь к датасету
dataset_path = 'yolo_dataset/dataset.yaml'

# Используем базовую модель для чистого обучения или best.pt если хотим дообучить
model = YOLO('yolov8n.pt') # или YOLO('best.pt')

print("Начинаем обучение...")
results = model.train(
    data=dataset_path,
    epochs=300,      # 300 эпох оптимально для начала (вместо 20000)
    imgsz=640,
    batch=16,        # Фиксированный батч для стабильности
    workers=2,       # 2 воркера оптимально для Colab (вместо 8)
    name='orange_colab',
    exist_ok=True,
    augment=True,
    box=7.5,         # Стандартный вес (вместо экстремальных 25.0)
    cls=0.5,
    dfl=1.5,
    patience=50,     # Ранняя остановка если нет улучшений
    optimizer='auto',
    project='/content/drive/MyDrive/YOLO_Orange_Sniper/runs' # Сохраняем сразу на Google Drive
)
print("Обучение завершено! Веса сохранены на Google Drive.")
"""

with open('train_colab.py', 'w') as f:
    f.write(train_script)

print("Скрипт train_colab.py создан!")

### 6. Запуск обучения
Запускаем наш оптимизированный скрипт. Результаты будут автоматически сохраняться на ваш Google Drive.

In [ ]:
!python train_colab.py

### 7. Тестирование модели (Inference)
Проверим, как работает обученная модель на тестовой картинке.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

# Загружаем лучшие веса с Google Drive
model = YOLO('/content/drive/MyDrive/YOLO_Orange_Sniper/runs/orange_colab/weights/best.pt')

# Укажите путь к тестовой картинке
test_img_path = 'yolo_dataset/images/val/test_image.jpg' # Замените на реальное имя файла

if os.path.exists(test_img_path):
    results = model(test_img_path)
    
    # Отображаем результат
    res_plotted = results[0].plot()
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()
else:
    print(f"Файл {test_img_path} не найден. Пожалуйста, укажите правильный путь к картинке.")